In [1]:
!pip install pandas


[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import re
import os
import json
from typing import List, Optional

In [3]:
MODEL_NAME = "gemma3_4b"
MODEL_NAME_CLEAN = MODEL_NAME.replace(':', '_')
BASE_PATH = f"../Output Performance Degradation Analysis/{MODEL_NAME_CLEAN}"

# Task configurati
TASK_NAMES = {
    1: "json_only",
    2: "json_translation", 
    3: "json_translation_sentiment",
    4: "json_translation_sentiment_ner"
}

# Colonne da mantenere
ALLOWED_COLUMNS = ["review", "sentiment", "entities", "json", "translate", "progressive_index", "output"]

print(f"✓ Configurazione completata:")
print(f"  - Modello: {MODEL_NAME}")
print(f"  - Percorso base: {BASE_PATH}")
print(f"  - Task configurati: {len(TASK_NAMES)}")
print(f"  - Colonne mantenute: {ALLOWED_COLUMNS}")

✓ Configurazione completata:
  - Modello: gemma3_4b
  - Percorso base: ../Output Performance Degradation Analysis/gemma3_4b
  - Task configurati: 4
  - Colonne mantenute: ['review', 'sentiment', 'entities', 'json', 'translate', 'progressive_index', 'output']


In [4]:
def extract_json_from_output(output: str) -> Optional[str]:
    """
    Estrae il JSON dall'output dell'LLM trovando tutto il contenuto tra { e }.
    
    Args:
        output (str): L'output grezzo dell'LLM
        
    Returns:
        str or None: Il JSON estratto come stringa, o None se non trovato
    """
    if pd.isna(output) or output == "$$":
        return None

    output = re.sub(pattern=r"(?<!\S)//.*?$", flags=re.MULTILINE, string=output, repl="")
    output = re.sub(pattern=r"```json\s*", string=output, repl="", flags=re.IGNORECASE)
    output = re.sub(pattern=r"```\s*$", string=output, repl="", flags=re.MULTILINE)
    
    # Trova la prima parentesi graffa di apertura
    start_index = output.find('{')
    if start_index == -1:
        return None
    
    # Trova l'ultima parentesi graffa di chiusura
    end_index = output.rfind('}')
    if end_index == -1 or end_index <= start_index:
        return None
    
    # Estrai tutto quello che sta tra le parentesi graffe (incluse)
    json_candidate = output[start_index:end_index + 1].strip()
    
    return json_candidate

print("✓ Funzione extract_json_from_output definita")

✓ Funzione extract_json_from_output definita


In [5]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pulisce il dataframe mantenendo solo le colonne necessarie e aggiungendo json_output.
    
    Args:
        df (pd.DataFrame): Il dataframe da pulire
        
    Returns:
        pd.DataFrame: Il dataframe pulito
    """
    # Mantieni solo le colonne necessarie
    columns_to_keep = [col for col in ALLOWED_COLUMNS if col in df.columns]
    df_cleaned = df[columns_to_keep].copy()
    
    # Aggiungi la colonna json_output
    df_cleaned["json_output"] = df_cleaned["output"].apply(extract_json_from_output)
    
    return df_cleaned

print("✓ Funzione clean_dataframe definita")

✓ Funzione clean_dataframe definita


In [6]:
def process_task_file(task_number: int) -> None:
    """
    Processa un singolo file di task.
    
    Args:
        task_number (int): Numero del task (1-4)
    """
    task_name = TASK_NAMES[task_number]
    
    # Percorsi file
    input_file = f"{BASE_PATH}/sampled_reviews_task_{task_number}_{task_name}_{MODEL_NAME_CLEAN}.csv"
    output_file = f"{BASE_PATH}_cleaned/sampled_reviews_task_{task_number}_{task_name}_{MODEL_NAME_CLEAN}_cleaned.csv"
    output_dir = os.path.dirname(output_file)
    os.makedirs(output_dir, exist_ok=True)

    print(f"Processando Task {task_number} ({task_name})...")
    
    # Verifica esistenza file
    if not os.path.exists(input_file):
        print(f"❌ File non trovato: {input_file}")
        return
    
    try:
        # Carica il dataframe
        df = pd.read_csv(input_file)
        print(f"✓ File caricato: {len(df)} righe")
        
        # Pulisci il dataframe
        df_cleaned = clean_dataframe(df)
        
        # Conta JSON validi estratti
        valid_json_count = df_cleaned["json_output"].notna().sum()
        total_rows = len(df_cleaned)
        
        print(f"✓ JSON validi estratti: {valid_json_count}/{total_rows} ({valid_json_count/total_rows*100:.1f}%)")
        
        # Salva il file pulito
        df_cleaned.to_csv(output_file, index=False)
        print(f"✓ File salvato: {output_file}")
        
    except Exception as e:
        print(f"❌ Errore nel processare {input_file}: {str(e)}")
    
    print("-" * 60)

print("✓ Funzione process_task_file definita")

✓ Funzione process_task_file definita


In [7]:
print("=" * 60)
print("VERIFICA CONFIGURAZIONE")
print("=" * 60)

# Verifica esistenza directory
if not os.path.exists(BASE_PATH):
    print(f"❌ Directory non trovata: {BASE_PATH}")
else:
    print(f"✓ Directory trovata: {BASE_PATH}")
    
    # Lista file nella directory
    files_in_dir = [f for f in os.listdir(BASE_PATH) if f.endswith('.csv')]
    print(f"✓ File CSV trovati: {len(files_in_dir)}")
    
    # Verifica esistenza file per ogni task
    for task_number, task_name in TASK_NAMES.items():
        input_file = f"sampled_reviews_task_{task_number}_{task_name}_{MODEL_NAME_CLEAN}.csv"
        if input_file in files_in_dir:
            print(f"✓ Task {task_number}: {input_file}")
        else:
            print(f"❌ Task {task_number}: {input_file} NON TROVATO")

VERIFICA CONFIGURAZIONE
✓ Directory trovata: ../Output Performance Degradation Analysis/gemma3_4b
✓ File CSV trovati: 4
✓ Task 1: sampled_reviews_task_1_json_only_gemma3_4b.csv
✓ Task 2: sampled_reviews_task_2_json_translation_gemma3_4b.csv
✓ Task 3: sampled_reviews_task_3_json_translation_sentiment_gemma3_4b.csv
✓ Task 4: sampled_reviews_task_4_json_translation_sentiment_ner_gemma3_4b.csv


In [8]:
print("=" * 60)
print("TASK 1: JSON Only")
print("=" * 60)

process_task_file(1)

TASK 1: JSON Only
Processando Task 1 (json_only)...
✓ File caricato: 500 righe
✓ JSON validi estratti: 500/500 (100.0%)
✓ File salvato: ../Output Performance Degradation Analysis/gemma3_4b_cleaned/sampled_reviews_task_1_json_only_gemma3_4b_cleaned.csv
------------------------------------------------------------


In [9]:
print("=" * 60)
print("TASK 2: JSON + Translation")
print("=" * 60)

process_task_file(2)

TASK 2: JSON + Translation
Processando Task 2 (json_translation)...
✓ File caricato: 500 righe
✓ JSON validi estratti: 497/500 (99.4%)
✓ File salvato: ../Output Performance Degradation Analysis/gemma3_4b_cleaned/sampled_reviews_task_2_json_translation_gemma3_4b_cleaned.csv
------------------------------------------------------------


In [10]:
print("=" * 60)
print("TASK 3: JSON + Translation + Sentiment")
print("=" * 60)

process_task_file(3)

TASK 3: JSON + Translation + Sentiment
Processando Task 3 (json_translation_sentiment)...
✓ File caricato: 500 righe
✓ JSON validi estratti: 499/500 (99.8%)
✓ File salvato: ../Output Performance Degradation Analysis/gemma3_4b_cleaned/sampled_reviews_task_3_json_translation_sentiment_gemma3_4b_cleaned.csv
------------------------------------------------------------


In [11]:
print("=" * 60)
print("TASK 4: JSON + Translation + Sentiment + NER")
print("=" * 60)

process_task_file(4)

TASK 4: JSON + Translation + Sentiment + NER
Processando Task 4 (json_translation_sentiment_ner)...
✓ File caricato: 500 righe
✓ JSON validi estratti: 500/500 (100.0%)
✓ File salvato: ../Output Performance Degradation Analysis/gemma3_4b_cleaned/sampled_reviews_task_4_json_translation_sentiment_ner_gemma3_4b_cleaned.csv
------------------------------------------------------------


In [12]:
print("=" * 60)
print("RIEPILOGO FINALE")
print("=" * 60)

total_valid_json = 0
total_rows = 0

for task_number, task_name in TASK_NAMES.items():
    output_file = f"{BASE_PATH}_cleaned/sampled_reviews_task_{task_number}_{task_name}_{MODEL_NAME_CLEAN}_cleaned.csv"
    
    if os.path.exists(output_file):
        df = pd.read_csv(output_file)
        valid_json_count = df["json_output"].notna().sum()
        rows_count = len(df)
        
        total_valid_json += valid_json_count
        total_rows += rows_count
        
        print(f"Task {task_number} ({task_name}):")
        print(f"  - Righe totali: {rows_count}")
        print(f"  - JSON validi: {valid_json_count} ({valid_json_count/rows_count*100:.1f}%)")
        print(f"  - File: {output_file}")
    else:
        print(f"❌ Task {task_number}: File cleaned non trovato")

print("-" * 60)
print(f"TOTALE:")
print(f"  - Righe processate: {total_rows}")
print(f"  - JSON validi estratti: {total_valid_json} ({total_valid_json/total_rows*100:.1f}%)")
print("✅ Processo completato!")

RIEPILOGO FINALE
Task 1 (json_only):
  - Righe totali: 500
  - JSON validi: 500 (100.0%)
  - File: ../Output Performance Degradation Analysis/gemma3_4b_cleaned/sampled_reviews_task_1_json_only_gemma3_4b_cleaned.csv
Task 2 (json_translation):
  - Righe totali: 500
  - JSON validi: 497 (99.4%)
  - File: ../Output Performance Degradation Analysis/gemma3_4b_cleaned/sampled_reviews_task_2_json_translation_gemma3_4b_cleaned.csv
Task 3 (json_translation_sentiment):
  - Righe totali: 500
  - JSON validi: 499 (99.8%)
  - File: ../Output Performance Degradation Analysis/gemma3_4b_cleaned/sampled_reviews_task_3_json_translation_sentiment_gemma3_4b_cleaned.csv
Task 4 (json_translation_sentiment_ner):
  - Righe totali: 500
  - JSON validi: 500 (100.0%)
  - File: ../Output Performance Degradation Analysis/gemma3_4b_cleaned/sampled_reviews_task_4_json_translation_sentiment_ner_gemma3_4b_cleaned.csv
------------------------------------------------------------
TOTALE:
  - Righe processate: 2000
  - JSO